<div dir="rtl">

# تحلیل داده‌های فروش و رفتار مشتریان فروشگاه اینترنتی Olist

**پروژه تحلیل داده با پایتون (Pandas — Matplotlib — Seaborn)**

---

## مرحله ۱: شناخت داده‌ها

پیش از هر تحلیلی باید بدانیم دقیقاً چه داده‌ای در اختیار داریم. در این مرحله هر جدول را بررسی می‌کنیم:

- هر جدول چند سطر و چند ستون دارد؟
- هر سطر نشان‌دهنده چیست؟
- نوع داده ستون‌ها چیست و آیا نوعی نامناسب دارند؟
- مقادیر گمشده و رکوردهای تکراری چقدر است؟
- کدام ستون‌ها برای برقراری ارتباط میان جداول استفاده می‌شوند؟

**نکته مهم:** در این مرحله هیچ داده‌ای پاک‌سازی نمی‌شود؛ فقط مشاهده و اندازه‌گیری می‌کنیم.

### درباره دیتاست

Olist یک پلتفرم تجارت الکترونیک (Marketplace) در برزیل است؛ فروشندگان مختلف محصولات خود را از طریق فروشگاه Olist به مشتریان سراسر برزیل می‌فروشند. داده‌های این پروژه شامل ۹ جدول است که سفارش‌ها، مشتریان، محصولات، فروشندگان، پرداخت‌ها، ارسال سفارش‌ها و نظرات مشتریان را توصیف می‌کنند.

### ساختار ارتباط جداول

```
مشتری (customers)
   │
   ▼
سفارش (orders)
   │
   ├──► پرداخت (payments)
   ├──► نظرات (reviews)
   ├──► اقلام سفارش (order_items) ──► فروشنده (sellers)
   │            │
   │            └──► محصول (products) ──► ترجمه دسته‌بندی (category_translation)
   │
   └──► جغرافیا (geolocation)   ◄── کد پستی مشتری و فروشنده
```

</div>

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 130)

DATA = 'data/raw/'

orders = pd.read_csv(DATA + 'olist_orders_dataset.csv')
customers = pd.read_csv(DATA + 'olist_customers_dataset.csv')
items = pd.read_csv(DATA + 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(DATA + 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA + 'olist_sellers_dataset.csv')
geolocation = pd.read_csv(DATA + 'olist_geolocation_dataset.csv')
category_translation = pd.read_csv(DATA + 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'order_items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation': geolocation,
    'category_translation': category_translation,
}

<div dir="rtl">

## ۱-۱. شکل کلی جداول

هر سطر هر جدول چه چیزی را توصیف می‌کند:

| جدول | هر سطر نشان‌دهنده |
|---|---|
| orders | یک سفارش ثبت‌شده توسط مشتری |
| customers | یک سفارش + اطلاعات مشتری آن سفارش |
| order_items | یک قلم (محصول) از یک سفارش |
| payments | یک تراکنش پرداخت |
| reviews | یک نظر ثبت‌شده برای یک سفارش |
| products | یک محصول یکتای فروخته‌شده |
| sellers | یک فروشنده فعال |
| geolocation | یک کد پستی + یک مختصات جغرافیایی |
| category_translation | یک جفت نام پرتغالی / انگلیسی دسته‌بندی |

</div>

In [2]:
overview = pd.DataFrame({
    'تعداد سطر': {k: len(v) for k, v in tables.items()},
    'تعداد ستون': {k: v.shape[1] for k, v in tables.items()},
    'تعداد سطر تکراری': {k: int(v.duplicated().sum()) for k, v in tables.items()},
})
overview

,تعداد سطر,تعداد ستون,تعداد سطر تکراری
orders,99441,8,0
customers,99441,5,0
order_items,112650,7,0
payments,103886,5,0
reviews,99224,7,0
products,32951,9,0
sellers,3095,4,0
geolocation,1000163,5,261831
category_translation,71,2,0


In [3]:
def profile(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'null': df.isna().sum(),
        'null_pct': (df.isna().mean() * 100).round(2),
        'nunique': df.nunique(dropna=False),
    })


for name, df in tables.items():
    print('=' * 60)
    print(name)
    print('=' * 60)
    display(profile(df))
    print()

orders


,dtype,null,null_pct,nunique
order_id,str,0,0.00,99441
customer_id,str,0,0.00,99441
order_status,str,0,0.00,8
order_purchase_timestamp,str,0,0.00,98875
order_approved_at,str,160,0.16,90734
order_delivered_carrier_date,str,1783,1.79,81019
order_delivered_customer_date,str,2965,2.98,95665
order_estimated_delivery_date,str,0,0.00,459



customers


,dtype,null,null_pct,nunique
customer_id,str,0,0.0,99441
customer_unique_id,str,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,str,0,0.0,4119
customer_state,str,0,0.0,27



order_items


...

,dtype,null,null_pct,nunique
seller_id,str,0,0.0,3095
seller_zip_code_prefix,int64,0,0.0,2246
seller_city,str,0,0.0,611
seller_state,str,0,0.0,23



geolocation


,dtype,null,null_pct,nunique
geolocation_zip_code_prefix,int64,0,0.0,19015
geolocation_lat,float64,0,0.0,717372
geolocation_lng,float64,0,0.0,717615
geolocation_city,str,0,0.0,8011
geolocation_state,str,0,0.0,27



category_translation


,dtype,null,null_pct,nunique
product_category_name,str,0,0.0,71
product_category_name_english,str,0,0.0,71


<div dir="rtl">

## ۱-۲. جدول سفارش‌ها (orders)

محور اصلی کل تحلیل؛ هر جدول دیگری در نهایت به این جدول وصل می‌شود.

**ستون‌های زمانی این جدول:**

- `order_purchase_timestamp`: لحظه ثبت سفارش توسط مشتری
- `order_approved_at`: تأیید پرداخت
- `order_delivered_carrier_date`: تحویل به پیک
- `order_delivered_customer_date`: تحویل به مشتری
- `order_estimated_delivery_date`: تاریخ تخمینی تحویل که به مشتری وعده داده شده

</div>

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
vc = orders['order_status'].value_counts(dropna=False)
pd.DataFrame({
    'تعداد': vc,
    'درصد': (vc / len(orders) * 100).round(2),
})

,تعداد,درصد
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


In [6]:
print('سفارش‌های delivered با تاریخ تحویل موجود:',
      ((orders['order_status'] == 'delivered') & orders['order_delivered_customer_date'].notna()).sum())
print('سفارش‌های delivered بدون تاریخ تحویل:',
      ((orders['order_status'] == 'delivered') & orders['order_delivered_customer_date'].isna()).sum())
print()
print('سفارش‌های غیر-delivered با تاریخ تحویل:',
      ((orders['order_status'] != 'delivered') & orders['order_delivered_customer_date'].notna()).sum())

سفارش‌های delivered با تاریخ تحویل موجود: 96470
سفارش‌های delivered بدون تاریخ تحویل: 8

سفارش‌های غیر-delivered با تاریخ تحویل: 6


<div dir="rtl">

### یافته‌های جدول سفارش‌ها

1. **۹۹,۴۴۱ سفارش** با ۸ ستون؛ `order_id` کاملاً یکتا است، یعنی سفارش تکراری نداریم.
2. ۹۷٪ سفارش‌ها `delivered` (تحویل‌شده) هستند؛ بقیه در وضعیت‌هایی مثل `shipped`، `canceled` و `unavailable` قرار دارند.
3. سه ستون تاریخی خالی دارند (`order_approved_at` با ۰.۱۶٪، `order_delivered_carrier_date` با ۱.۷۹٪ و `order_delivered_customer_date` با ۲.۹۸٪). این خالی‌ها منطقی‌اند: سفارشی که هنوز تحویل نشده، تاریخ تحویل ندارد.
4. **۲ سفارش با وضعیت delivered وجود دارند که تاریخ تحویل ندارند** — ناهماهنگی داده‌ای که باید در مرحله پاک‌سازی بررسی شود.
5. همه ستون‌های تاریخی به‌صورت متن (str) ذخیره شده‌اند و در مرحله پاک‌سازی باید به datetime تبدیل شوند.
6. **نکته کلیدی:** ستون `customer_id` در این جدول برای هر سفارش یک مقدار یکتا دارد؛ یعنی شناسه مشتری در قالب همان سفارش است، نه شناسه ثابت مشتری. شناسه واقعی مشتری در جدول customers با نام `customer_unique_id` نگهداری می‌شود (در بخش بعد اثبات می‌شود).

</div>

<div dir="rtl">

## ۱-۳. جدول مشتریان (customers)

</div>

In [7]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [8]:
print('تعداد سطر جدول:                ', len(customers))
print('customer_id یکتا:               ', customers['customer_id'].nunique())
print('customer_unique_id یکتا:        ', customers['customer_unique_id'].nunique())
print('اختلاف (سفارش‌های تکراری مشتریان واقعی):',
      len(customers) - customers['customer_unique_id'].nunique())

تعداد سطر جدول:                 99441
customer_id یکتا:                99441
customer_unique_id یکتا:         96096
اختلاف (سفارش‌های تکراری مشتریان واقعی): 3345


In [9]:
orders_per_customer = customers['customer_unique_id'].value_counts()
vc = orders_per_customer.value_counts().sort_index()
pd.DataFrame({
    'تعداد سفارش': vc.index,
    'تعداد مشتریان': vc.values,
    'درصد مشتریان': (vc.values / len(orders_per_customer) * 100).round(2),
}).head(8)

,تعداد سفارش,تعداد مشتریان,درصد مشتریان
0,1,93099,96.88
1,2,2745,2.86
2,3,203,0.21
3,4,30,0.03
4,5,8,0.01
5,6,6,0.01
6,7,3,0.00
7,9,1,0.00


In [10]:
vc = customers['customer_state'].value_counts()
state_customers = pd.DataFrame({
    'تعداد مشتری': vc,
    'درصد': (vc / len(customers) * 100).round(2),
})
display(state_customers.head(10))
print('تعداد کل ایالت‌ها:', customers['customer_state'].nunique())
print('تعداد کل شهرها:  ', customers['customer_city'].nunique())
vc = customers['customer_city'].value_counts()
display(pd.DataFrame({'تعداد مشتری': vc, 'درصد': (vc / len(customers) * 100).round(2)}).head(10))

,تعداد مشتری,درصد
customer_state,,
SP,41746,41.98
RJ,12852,12.92
MG,11635,11.70
RS,5466,5.50
PR,5045,5.07
SC,3637,3.66
BA,3380,3.40
DF,2140,2.15
ES,2033,2.04


تعداد کل ایالت‌ها: 27
تعداد کل شهرها:   4119


,تعداد مشتری,درصد
customer_city,,
sao paulo,15540,15.63
rio de janeiro,6882,6.92
belo horizonte,2773,2.79
brasilia,2131,2.14
curitiba,1521,1.53
campinas,1444,1.45
porto alegre,1379,1.39
salvador,1245,1.25
guarulhos,1189,1.20


<div dir="rtl">

### یافته‌های جدول مشتریان

1. جدول ۹۹,۴۴۱ سطر دارد ولی فقط **۹۶,۰۹۶ مشتری یکتا** (`customer_unique_id`). یعنی حدود ۳,۳۴۵ سفارش متعلق به مشتریانی است که قبلاً هم خرید کرده‌اند.
2. **۹۶.۸۸٪ مشتریان فقط یک بار** خرید کرده‌اند؛ میانگین سفارش هر مشتری واقعی حدود ۱.۰۳ است. رفتار غالب، خرید تک‌باره است.
3. بازار شدیداً متمرکز است: ایالت‌های SP با ۴۲٪، RJ با ۱۲.۹٪ و MG با ۱۱.۷٪ مجموعاً حدود ۶۶.۶٪ مشتریان را دارند. شهر سائوپائولو به‌تنهایی ۱۵.۶٪ مشتریان است.
4. جدول هیچ مقدار خالی ندارد.

</div>

<div dir="rtl">

## ۱-۴. جدول اقلام سفارش (order_items)

قلب مالی دیتاست: ستون `price` مبنای محاسبه مبلغ فروش در همه تحلیل‌های بعدی است.

</div>

In [11]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [12]:
items_per_order = items.groupby('order_id').size()
vc = items_per_order.value_counts().sort_index()
display(pd.DataFrame({
    'تعداد اقلام': vc.index,
    'تعداد سفارش‌ها': vc.values,
    'درصد سفارش‌ها': (vc.values / len(items_per_order) * 100).round(2),
}).head(8))
print('سفارش‌های پوشش‌داده‌شده:', items['order_id'].nunique())
print('میانگین اقلام هر سفارش: ', round(items_per_order.mean(), 4))
print('بیشترین اقلام یک سفارش: ', items_per_order.max())
print('تکراری با کلید (order_id, order_item_id):',
      items.duplicated(subset=['order_id', 'order_item_id']).sum())

,تعداد اقلام,تعداد سفارش‌ها,درصد سفارش‌ها
0,1,88863,90.06
1,2,7516,7.62
2,3,1322,1.34
3,4,505,0.51
4,5,204,0.21
5,6,198,0.20
6,7,22,0.02
7,8,8,0.01


سفارش‌های پوشش‌داده‌شده: 98666
میانگین اقلام هر سفارش:  1.1417
بیشترین اقلام یک سفارش:  21
تکراری با کلید (order_id, order_item_id): 0


In [13]:
display(items[['price', 'freight_value']].describe().round(2))
print('اقلام با قیمت صفر:      ', (items['price'] == 0).sum())
print('اقلام با حمل صفر:       ', (items['freight_value'] == 0).sum())
print('صدک ۹۹ قیمت:            ', round(items['price'].quantile(0.99), 2))
print('اقلام بالاتر از صدک ۹۹: ', (items['price'] > items['price'].quantile(0.99)).sum())

,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,6735.00,409.68


اقلام با قیمت صفر:       0
اقلام با حمل صفر:        383
صدک ۹۹ قیمت:             890.0
اقلام بالاتر از صدک ۹۹:  1117


<div dir="rtl">

### یافته‌های جدول اقلام سفارش

1. **۱۱۲,۶۵۰ قلم** برای **۹۸,۶۶۶ سفارش** — یعنی هر سطر یک قلم است، نه یک سفارش. هر جا خواستیم تعداد سفارش حساب کنیم باید `order_id` را یکتا شمارش کنیم.
2. حدود ۷۷۵ سفارش جدول orders در این جدول ردیف ندارند (عمدتاً سفارش‌های لغوشده که محصولی ندارند).
3. **۹۰٪ سفارش‌ها فقط یک قلم** دارند؛ میانگین ۱.۱۴ قلم هر سفارش و بیشترین مقدار ۲۱ قلم.
4. ترکیب `(order_id, order_item_id)` کلید یکتای جدول است.
5. توزیع قیمت چوله به راست است: میانگین ۱۲۰.۶۵ ولی میانه ۷۴.۹۹ — گزارش میانگین به‌تنهایی تصویر کاملی نمی‌دهد.
6. ۳۸۳ قلم با هزینه حمل صفر (احتمال ارسال رایگان یا خطای ثبت) برای بررسی در مرحله پاک‌سازی.

</div>

<div dir="rtl">

## ۱-۵. جدول پرداخت‌ها (payments)

</div>

In [14]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [15]:
tx_per_order = payments.groupby('order_id').size()
print('تعداد سطر (تراکنش): ', len(payments))
print('سفارش‌های یکتای پرداخت:', payments['order_id'].nunique())
print('سفارش‌های چند-تراکنشی: ', (tx_per_order > 1).sum())
print()
vc = tx_per_order.value_counts().sort_index()
display(pd.DataFrame({'تعداد تراکنش': vc.index, 'تعداد سفارش‌ها': vc.values}).head(6))

sample_id = payments.groupby('order_id').filter(lambda x: len(x) > 1)['order_id'].iloc[0]
display(payments[payments['order_id'] == sample_id])

تعداد سطر (تراکنش):  103886
سفارش‌های یکتای پرداخت: 99440
سفارش‌های چند-تراکنشی:  2961



,تعداد تراکنش,تعداد سفارش‌ها
0,1,96479
1,2,2382
2,3,301
3,4,108
4,5,52
5,6,36


,order_id,payment_sequential,payment_type,payment_installments,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,1,45.17
57742,5cfd514482e22bc992e7693f0e3e8df7,1,credit_card,4,665.41


In [16]:
vc = payments['payment_type'].value_counts()
display(pd.DataFrame({
    'تعداد تراکنش': vc,
    'درصد تراکنش': (vc / len(payments) * 100).round(2),
}))

g = payments.groupby('payment_type')['payment_value'].agg(['count', 'sum', 'mean', 'median']).round(2)
g.columns = ['تعداد', 'مجموع مبلغ', 'میانگین', 'میانه']
display(g)

vc = payments['payment_installments'].value_counts().sort_index()
display(pd.DataFrame({
    'تعداد اقساط': vc.index,
    'تعداد تراکنش': vc.values,
    'درصد': (vc.values / len(payments) * 100).round(2),
}).head(12))

print('پرداخت‌های با مبلغ صفر: ', (payments['payment_value'] == 0).sum())
print('پرداخت‌های با ۰ قسط:    ', (payments['payment_installments'] == 0).sum())

,تعداد تراکنش,درصد تراکنش
payment_type,,
credit_card,76795,73.92
boleto,19784,19.04
voucher,5775,5.56
debit_card,1529,1.47
not_defined,3,0.00


,تعداد,مجموع مبلغ,میانگین,میانه
payment_type,,,,
boleto,19784,2869361.27,145.03,93.89
credit_card,76795,12542084.19,163.32,106.87
debit_card,1529,217989.79,142.57,89.30
not_defined,3,0.00,0.00,0.00
voucher,5775,379436.87,65.70,39.28


,تعداد اقساط,تعداد تراکنش,درصد
0,0,2,0.00
1,1,52546,50.58
2,2,12413,11.95
3,3,10461,10.07
4,4,7098,6.83
5,5,5239,5.04
6,6,3920,3.77
7,7,1626,1.57
8,8,4268,4.11
9,9,644,0.62


پرداخت‌های با مبلغ صفر:  9
پرداخت‌های با ۰ قسط:     2


<div dir="rtl">

### یافته‌های جدول پرداخت‌ها

1. **۱۰۳,۸۸۶ تراکنش** برای **۹۹,۴۴۰ سفارش** — حدود **۲,۹۶۱ سفارش چند تراکنش** دارند (مثلاً ترکیب ووچر با کارت اعتباری). بنابراین هر سطر پرداخت لزوماً یک سفارش یکتا نیست.
2. **۱ سفارش در این جدول وجود ندارد** (۹۹,۴۴۰ در برابر ۹۹,۴۴۱ سفارش جدول orders) — باید در مرحله پاک‌سازی شناسایی شود.
3. کارت اعتباری با ۷۴٪ تراکنش و ۷۸٪ مبلغ، روش غالب است؛ سند پرداخت `boleto` حدود ۱۹٪ تراکنش‌ها را دارد.
4. نیمی از پرداخت‌ها تک‌قسط هستند و قله‌های توزیع (۶، ۸ و ۱۰ قسط) با بسته‌های استاندارد بانکی منطبق است.
5. ۹ پرداخت با مبلغ صفر و ۲ پرداخت با صفر قسط برای بررسی در مرحله پاک‌سازی ثبت شد.
6. برای محاسبه میانگین مبلغ هر سفارش باید ابتدا `groupby(order_id)` انجام شود، وگرنه سفارش‌های چند-تراکنشی چند بار شمرده می‌شوند.

</div>

<div dir="rtl">

## ۱-۶. جدول نظرات مشتریان (reviews)

</div>

In [17]:
reviews.head().drop(columns=['review_comment_message'])

,review_id,order_id,review_score,review_comment_title,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,2018-03-01 00:00:00,2018-03-02 10:26:53


In [18]:
rev_per_order = reviews.groupby('order_id').size()
print('تعداد سطر (نظر):       ', len(reviews))
print('سفارش‌های نظردار:      ', reviews['order_id'].nunique())
print('review_id تکراری:      ', reviews['review_id'].duplicated().sum())
print()
vc = rev_per_order.value_counts().sort_index()
display(pd.DataFrame({'تعداد نظر': vc.index, 'تعداد سفارش‌ها': vc.values}).head(5))

تعداد سطر (نظر):        99224
سفارش‌های نظردار:       98673


review_id تکراری:       814



,تعداد نظر,تعداد سفارش‌ها
0,1,98126
1,2,543
2,3,4


In [19]:
vc = reviews['review_score'].value_counts().sort_index()
display(pd.DataFrame({
    'امتیاز': vc.index,
    'تعداد': vc.values,
    'درصد': (vc.values / len(reviews) * 100).round(2),
}))
print('میانگین امتیاز:          ', round(reviews['review_score'].mean(), 4))
print('میانه امتیاز:            ', reviews['review_score'].median())
print('درصد رضایت (۴ یا ۵):    ', round(reviews['review_score'].isin([4, 5]).mean() * 100, 2))
print('درصد نارضایتی (۱ یا ۲): ', round(reviews['review_score'].isin([1, 2]).mean() * 100, 2))
print()
print('نظرات با متن:   ', reviews['review_comment_message'].notna().sum(),
      f"({reviews['review_comment_message'].notna().mean() * 100:.2f}%)")
print('نظرات با عنوان: ', reviews['review_comment_title'].notna().sum())

,امتیاز,تعداد,درصد
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


میانگین امتیاز:           4.0864
میانه امتیاز:             5.0
درصد رضایت (۴ یا ۵):     77.07
درصد نارضایتی (۱ یا ۲):  14.69

نظرات با متن:    40977 (41.30%)
نظرات با عنوان:  11568


<div dir="rtl">

### یافته‌های جدول نظرات

1. **۹۹,۲۲۴ نظر** برای **۹۸,۶۷۳ سفارش**؛ ۵۴۷ سفارش دو نظر و ۴ سفارش سه نظر دارند و ۸۱۴ `review_id` تکراری وجود دارد. پس `review_id` کلید قابل اعتمادی نیست و برای شمارش سفارش‌ها باید `order_id` را یکتا در نظر گرفت.
2. حدود ۷۶۸ سفارش جدول orders هیچ نظری ندارند؛ در تحلیل رضایت باید مشخص شود مخرج محاسبه کدام است (همه سفارش‌ها یا فقط سفارش‌های نظردار).
3. میانگین امتیاز **۴.۰۹** است ولی توزیع دوقلو دارد: ۵۷.۸٪ امتیاز ۵، و امتیاز ۱ (۱۱.۵٪) بیشتر از امتیاز ۲ و ۳ است. گزارش صرف میانگین، وضعیت را بهتر از واقعیت نشان می‌دهد.
4. **۷۷.۰۷٪ نظرات رضایت‌مند** (۴ یا ۵) و **۱۴.۶۹٪ ناراضی** (۱ یا ۲) هستند.
5. ۵۸.۷٪ نظرات متن ندارند و فقط ۱۱.۷٪ عنوان دارند — این رفتار عادی کاربر است (ستاره زدن بدون متن)، نه داده گم‌شده؛ بنابراین این ستون‌ها حذف نمی‌شوند.

</div>

<div dir="rtl">

## ۱-۷. جدول محصولات (products)

</div>

In [20]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [21]:
print('دسته‌بندی‌های یکتا (پرتغالی):', products['product_category_name'].nunique())
print('محصولات بدون دسته‌بندی:      ', products['product_category_name'].isna().sum())
print()
vc = products['product_category_name'].value_counts()
display(pd.DataFrame({'تعداد محصول': vc}).head(10))
display(pd.DataFrame({'تعداد محصول': vc}).tail(5))

dim_cols = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
            'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
print('محصولاتی که هر ۷ ستون ویژگی خالی دارند:', products[dim_cols].isna().all(axis=1).sum())
print('محصولاتی که حداقل یک ویژگی خالی دارند: ', products[dim_cols].isna().any(axis=1).sum())

دسته‌بندی‌های یکتا (پرتغالی): 73
محصولات بدون دسته‌بندی:       610



,تعداد محصول
product_category_name,
cama_mesa_banho,3029
esporte_lazer,2867
moveis_decoracao,2657
beleza_saude,2444
utilidades_domesticas,2335
automotivo,1900
informatica_acessorios,1639
brinquedos,1411
relogios_presentes,1329


,تعداد محصول
product_category_name,
fashion_roupa_infanto_juvenil,5
casa_conforto_2,5
pc_gamer,3
seguros_e_servicos,2
cds_dvds_musicais,1


محصولاتی که هر ۷ ستون ویژگی خالی دارند: 1
محصولاتی که حداقل یک ویژگی خالی دارند:  611


In [22]:
display(products[['product_weight_g', 'product_length_cm', 'product_height_cm',
                  'product_width_cm', 'product_photos_qty']].describe().round(2))
print('وزن صفر:            ', (products['product_weight_g'] == 0).sum())
print('وزن بالای ۳۰ کیلو:  ', (products['product_weight_g'] > 30000).sum())

,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_photos_qty
count,32949.00,32949.00,32949.00,32949.00,32341.00
mean,2276.47,30.82,16.94,23.20,2.19
std,4282.04,16.91,13.64,12.08,1.74
min,0.00,7.00,2.00,6.00,1.00
25%,300.00,18.00,8.00,15.00,1.00
50%,700.00,25.00,13.00,20.00,1.00
75%,1900.00,38.00,21.00,30.00,3.00
max,40425.00,105.00,105.00,118.00,20.00


وزن صفر:             4
وزن بالای ۳۰ کیلو:   1


<div dir="rtl">

### یافته‌های جدول محصولات

1. **۳۲,۹۵۱ محصول** در ۷۳ دسته‌بندی پرتغالی؛ `product_id` کاملاً یکتا است.
2. **۶۱۰ محصول (۱.۸۵٪) دسته‌بندی ندارند** و دقیقاً همان محصولات، ستون‌های عنوان و توضیح و تعداد عکس را هم خالی دارند. در مقابل فقط ۲ محصول اطلاعات فیزیکی (وزن و ابعاد) ندارند. این دو گروه تقریباً همپوشانی ندارند.
3. تعداد محصول هر دسته‌بندی معیار فروش نیست؛ فروش واقعی باید از جدول order_items محاسبه شود.
4. توزیع وزن چوله به راست است (میانگین ۲.۲ کیلوگرم در برابر میانه ۰.۷ کیلوگرم) و ۴ محصول وزن صفر دارند.

</div>

<div dir="rtl">

## ۱-۸. جدول فروشندگان (sellers)

</div>

In [23]:
sellers_in_items = items['seller_id'].nunique()
print('تعداد فروشندگان:                          ', len(sellers))
print('seller_id یکتا:                           ', sellers['seller_id'].nunique())
print('فروشندگانی که در جدول اقلام فروش داشته‌اند:', sellers_in_items)
print('فروشنده بدون فروش:                        ', len(sellers) - sellers_in_items)
print('فروشنده یتیم (در اقلام هست، در این جدول نه):',
      len(set(items['seller_id'].unique()) - set(sellers['seller_id'].unique())))
print()
vc = sellers['seller_state'].value_counts()
display(pd.DataFrame({'تعداد فروشنده': vc, 'درصد': (vc / len(sellers) * 100).round(2)}).head(8))
vc = sellers['seller_city'].value_counts()
display(pd.DataFrame({'تعداد فروشنده': vc, 'درصد': (vc / len(sellers) * 100).round(2)}).head(8))

تعداد فروشندگان:                           3095
seller_id یکتا:                            3095
فروشندگانی که در جدول اقلام فروش داشته‌اند: 3095
فروشنده بدون فروش:                         0
فروشنده یتیم (در اقلام هست، در این جدول نه): 0



,تعداد فروشنده,درصد
seller_state,,
SP,1849,59.74
PR,349,11.28
MG,244,7.88
SC,190,6.14
RJ,171,5.53
RS,129,4.17
GO,40,1.29
DF,30,0.97


,تعداد فروشنده,درصد
seller_city,,
sao paulo,694,22.42
curitiba,127,4.10
rio de janeiro,96,3.10
belo horizonte,68,2.20
ribeirao preto,52,1.68
guarulhos,50,1.62
ibitinga,49,1.58
santo andre,45,1.45


<div dir="rtl">

### یافته‌های جدول فروشندگان

1. **۳,۰۹۵ فروشنده** در ۲۳ ایالت و ۶۱۱ شهر؛ تمیزترین جدول دیتاست: بدون مقدار خالی، بدون تکراری، و همه فروشندگان در جدول اقلام فروش داشته‌اند (هیچ فروشنده یتیمی نیست).
2. برخلاف جدول مشتریان، فروشندگان فقط یک شناسه یکتا دارند و مشکل چند-سفارشی‌بودن ندارند.
3. **تمرکز عرضه بیشتر از تقاضا در SP**: ۵۹.۷٪ فروشندگان در برابر ۴۲٪ مشتری. برعکس در **RJ**: فقط ۵.۵٪ فروشندگان در برابر ۱۲.۹٪ مشتری — یعنی سفارش‌های ایالت ریو احتمالاً از فروشنده‌های ایالت‌های دیگر ارسال می‌شوند که می‌تواند با فاصله حمل و تأخیر تحویل مرتبط باشد (در مرحله تحلیل ارسال بررسی می‌شود).
4. ایالت PR با ۱۱.۳٪ فروشنده و فقط ۵.۱٪ مشتری، نقش قطب تأمین را دارد.

</div>

<div dir="rtl">

## ۱-۹. جدول جغرافیا (geolocation)

بزرگ‌ترین جدول دیتاست (~۱ میلیون سطر)؛ کد پستی را به مختصات جغرافیایی تبدیل می‌کند.

</div>

In [24]:
coords_per_zip = geolocation.groupby('geolocation_zip_code_prefix').size()
print('تعداد سطر:                    ', len(geolocation))
print('کد پستی یکتا:                 ', geolocation['geolocation_zip_code_prefix'].nunique())
print('سطرهای کاملاً تکراری:         ', geolocation.duplicated().sum())
print('میانگین مختصات هر کد پستی:    ', round(coords_per_zip.mean(), 2))
print('بیشترین مختصات برای یک کد پستی:', coords_per_zip.max())

تعداد سطر:                    

 1000163
کد پستی یکتا:                  19015


سطرهای کاملاً تکراری:          261831
میانگین مختصات هر کد پستی:     52.6
بیشترین مختصات برای یک کد پستی: 1146


In [25]:
display(geolocation[['geolocation_lat', 'geolocation_lng']].describe().round(4))
bad_lat = ((geolocation['geolocation_lat'] < -40) | (geolocation['geolocation_lat'] > 10)).sum()
bad_lng = ((geolocation['geolocation_lng'] < -80) | (geolocation['geolocation_lng'] > -30)).sum()
print('مختصات خارج از محدوده برزیل (عرض):', bad_lat)
print('مختصات خارج از محدوده برزیل (طول):', bad_lng)

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117620e+01,-4.639050e+01
std,5.715900e+00,4.269700e+00
min,-3.660540e+01,-1.014668e+02
25%,-2.360350e+01,-4.857320e+01
50%,-2.291940e+01,-4.663790e+01
75%,-1.997960e+01,-4.376770e+01
max,4.506590e+01,1.211054e+02


مختصات خارج از محدوده برزیل (عرض): 26
مختصات خارج از محدوده برزیل (طول): 26


<div dir="rtl">

### یافته‌های جدول جغرافیا

1. **۱,۰۰۰,۱۶۳ سطر** برای ۱۹,۰۱۵ کد پستی — رابطه «یک-به-چند»: هر کد پستی به‌طور میانگین ۵۲ مختصات دارد چون یک منطقه پستی چند کیلومتر وسعت دارد. برای استفاده در تحلیل باید میانگین مختصات هر کد پستی (مرکز ثقل) محاسبه شود.
2. **۲۶.۲٪ سطرها کاملاً تکراری** هستند؛ این تکراری‌ها اگر حذف نشوند، محاسبه مرکز ثقل هر کد پستی را منحرف می‌کنند.
3. میانه مختصات (عرض ۲۲.۹- و طول ۴۶.۶-) دقیقاً روی سائوپائولو است، ولی **۲۶ سطر مختصات غیرممکن** (خارج از برزیل) وجود دارد.
4. نام ستون کد پستی در این جدول با جداول مشتریان و فروشندگان متفاوت است (`geolocation_zip_code_prefix` در برابر `customer_zip_code_prefix`) — در مرحله ترکیب جداول باید مراقب نام ستون‌ها بود.

</div>

<div dir="rtl">

## ۱-۱۰. جدول ترجمه دسته‌بندی محصولات

</div>

In [26]:
display(category_translation.head(10))
print('نام پرتغالی یکتا:', category_translation['product_category_name'].nunique())
print('نام انگلیسی یکتا:', category_translation['product_category_name_english'].nunique())
print('سطر تکراری:      ', category_translation.duplicated().sum())
print()
cats_products = set(products['product_category_name'].dropna().unique())
cats_translation = set(category_translation['product_category_name'].dropna().unique())
only_products = cats_products - cats_translation
only_translation = cats_translation - cats_products
print('دسته‌بندی‌های محصولات بدون ترجمه:', sorted(only_products))
print('ترجمه‌های بلااستفاده:            ', sorted(only_translation))

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


نام پرتغالی یکتا: 71
نام انگلیسی یکتا: 71
سطر تکراری:       0

دسته‌بندی‌های محصولات بدون ترجمه: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
ترجمه‌های بلااستفاده:             []


<div dir="rtl">

### یافته‌های جدول ترجمه

1. **۷۱ جفت ترجمه** پرتغالی-انگلیسی، کاملاً یک‌به‌یک و بدون خالی یا تکراری.
2. **۲ دسته‌بندی ترجمه نشده‌اند**: `pc_gamer` (۳ محصول) و `portateis_cozinha_e_preparadores_de_alimentos` (۱۰ محصول) — مجموعاً ۱۳ محصول (۰.۰۴٪ کل). در مرحله پاک‌سازی باید تصمیم گرفته شود حذف شوند یا برچسب «سایر» بگیرند.

</div>

<div dir="rtl">

## ۱-۱۱. بررسی ارتباط میان جداول

حالا بررسی می‌کنیم کلیدهای مشترک آیا واقعاً جداول را به هم وصل می‌کنند یا داده یتیم داریم.

</div>

In [27]:
print('orders  → customers :', len(set(orders['customer_id']) - set(customers['customer_id'])), 'سفارش بدون مشتری')
print('orders  → items     :', len(set(items['order_id']) - set(orders['order_id'])), 'قلام بدون سفارش')
print('orders  → payments  :', len(set(payments['order_id']) - set(orders['order_id'])), 'پرداخت بدون سفارش')
print('orders  → reviews   :', len(set(reviews['order_id']) - set(orders['order_id'])), 'نظر بدون سفارش')
print('items   → products  :', len(set(items['product_id']) - set(products['product_id'])), 'قلام بدون محصول')
print('items   → sellers   :', len(set(items['seller_id']) - set(sellers['seller_id'])), 'قلام بدون فروشنده')
print()
print('سفارش‌های بدون قلم:    ', len(set(orders['order_id']) - set(items['order_id'])))
print('سفارش‌های بدون پرداخت: ', len(set(orders['order_id']) - set(payments['order_id'])))
print('سفارش‌های بدون نظر:    ', len(set(orders['order_id']) - set(reviews['order_id'])))

orders  → customers : 0 سفارش بدون مشتری


orders  → items     : 0 قلام بدون سفارش


orders  → payments  : 0 پرداخت بدون سفارش


orders  → reviews   : 0 نظر بدون سفارش
items   → products  : 0 قلام بدون محصول
items   → sellers   : 0 قلام بدون فروشنده



سفارش‌های بدون قلم:     775


سفارش‌های بدون پرداخت:  1


سفارش‌های بدون نظر:     768


In [28]:
geo_zips = set(geolocation['geolocation_zip_code_prefix'])
cust_zip_covered = customers['customer_zip_code_prefix'].isin(geo_zips).mean() * 100
seller_zip_covered = sellers['seller_zip_code_prefix'].isin(geo_zips).mean() * 100
print(f'پوشش کد پستی مشتریان در جدول جغرافیا: {cust_zip_covered:.2f}%')
print(f'پوشش کد پستی فروشندگان در جدول جغرافیا: {seller_zip_covered:.2f}%')

پوشش کد پستی مشتریان در جدول جغرافیا: 99.72%
پوشش کد پستی فروشندگان در جدول جغرافیا: 99.77%


<div dir="rtl">

### نتیجه بررسی ارتباط جداول

1. کلیدهای اصلی (`customer_id`، `order_id`، `product_id`، `seller_id`) سالم هستند؛ هیچ قلم، پرداخت یا نظر یتیمی وجود ندارد.
2. حدود ۷۷۵ سفارش قلم ندارند، ۱ سفارش پرداخت ندارد و حدود ۷۶۸ سفارش نظر ندارند — همگی سفارش‌هایی با وضعیت‌های غیر تحویل‌شده.
3. کدهای پستی مشتریان و فروشندگان تقریباً به‌طور کامل در جدول جغرافیا پوشش داده شده‌اند (اختلاف جزئی به دلیل کدهای پستی بدون مختصات).

</div>

<div dir="rtl">

## ۱-۱۲. جمع‌بندی مرحله شناخت داده

### ساختار کلی داده‌ها در چند خط

دیتاست Olist شامل ۹ جدول رابطه‌ای است که حول محور **سفارش** سازمان یافته‌اند: هر سفارش یک ردیف در جدول orders دارد و از طریق `order_id` به پرداخت‌ها، نظرات و اقلام سفارش وصل می‌شود. هر قلم سفارش خودش دو رابطه دیگر دارد: به محصول و به فروشنده. جدول مشتریان از طریق `customer_id` به سفارش‌ها متصل است و شناسه واقعی مشتری را با `customer_unique_id` نگه می‌دارد. جدول جغرافیا از طریق کد پستی، به مشتریان و فروشندگان موقعیت مکانی می‌دهد و جدول ترجمه، دسته‌بندی‌های پرتغالی محصولات را به انگلیسی تبدیل می‌کند.

نکته محوری این دیتاست تفاوت میان «تعداد سطر» و «تعداد واحد واقعی» است: جدول پرداخت‌ها هر تراکنش را یک سطر می‌داند و جدول نظرات هر نظر را؛ بنابراین در همه محاسبات باید تعداد سفارش‌های یکتا از `order_id` شمارش شود، نه تعداد سطرها.

### فهرست مشکلات شناسایی‌شده برای مرحله پاک‌سازی

| # | مشکل | مقدار |
|---|---|---|
| ۱ | ستون‌های تاریخی به‌صورت متن ذخیره شده‌اند | ۸ ستون در orders + ۲ ستون در reviews |
| ۲ | سفارش‌های delivered بدون تاریخ تحویل | ۲ |
| ۳ | سفارش بدون هیچ پرداختی | ۱ |
| ۴ | پرداخت با مبلغ صفر / صفر قسط | ۹ / ۲ |
| ۵ | review_id تکراری | ۸۱۴ |
| ۶ | محصولات بدون دسته‌بندی | ۶۱۰ |
| ۷ | دسته‌بندی‌های بدون ترجمه (۱۳ محصول) | ۲ |
| ۸ | محصولات با وزن صفر | ۴ |
| ۹ | سطرهای تکراری جدول جغرافیا | ۲۶۱,۸۳۱ (۲۶.۲٪) |
| ۱۰ | مختصات خارج از محدوده برزیل | ۲۶ |
| ۱۱ | قلم‌هایی با هزینه حمل صفر | ۳۸۳ |

**مرحله بعد:** پاک‌سازی و آماده‌سازی داده‌ها (تبدیل تاریخ‌ها، تصمیم درباره مقادیر گمشده، ساخت ستون‌های مشتق‌شده مثل مدت تحویل و روز تأخیر).

</div>